ARTI308 - Machine Learning

# Project Notebook 3: Feature Engineering and Modeling (Classification)
## Heart Disease Prediction

**Goal:** build baseline classifiers that predict `HeartDisease` (0 / 1), compare them, and see how feature engineering affects the result.

### Why recall matters here
In a medical screening problem there are two kinds of mistakes:
- **False positive:** a healthy patient is flagged -> an extra check-up.
- **False negative:** a sick patient is missed -> much more dangerous.

So besides accuracy we pay special attention to **recall** of the disease class: out of all real patients, how many did the model find?

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the cleaned dataset
This file was created at the end of notebook 2. It still contains `NaN` in `Cholesterol` and `RestingBP` (the former hidden zeros) - we will impute them inside the pipeline.

In [ ]:
df = pd.read_csv("../data/processed/heart_clean.csv")
print("Shape:", df.shape)
print()
print(df.isna().sum()[df.isna().sum() > 0])
df.head()

**Observation:**

*(Write what you notice after running the cell above.)*

## 3. Leakage awareness (important)
We must not use information that would not be available at prediction time, and we must not let the **test set influence preprocessing**.

- Here all input features are measurements available during the consultation, and there are no ID or time columns, so no feature has to be dropped.
- The real leakage risk is in preprocessing: if we computed the median or the scaler on the whole dataset, information from the test set would leak into training.
- Therefore: **split first, then fit the imputer / scaler / encoder on the training set only.** A scikit-learn `Pipeline` does this automatically.

## 4. Feature engineering
### 4.1 Discretization: `age_group`
Convert the continuous `Age` into understandable groups. This can help models capture non-linear effects and improves interpretability.

In [ ]:
df_fe = df.copy()

df_fe["age_group"] = pd.cut(
    df_fe["Age"],
    bins=[0, 40, 50, 60, 200],
    labels=["under_40", "40_49", "50_59", "60_plus"]
).astype(str)

df_fe[["Age", "age_group"]].head(10)

**Observation:**

*(Write what you notice after running the cell above.)*

### 4.2 Domain feature: `maxhr_pct`
A common rule of thumb for the maximum heart rate expected for a person is `220 - Age`.
The ratio `MaxHR / (220 - Age)` tells how close the patient got to the expected maximum, which is more informative than raw `MaxHR` alone.

In [ ]:
df_fe["maxhr_pct"] = df_fe["MaxHR"] / (220 - df_fe["Age"])
df_fe[["Age", "MaxHR", "maxhr_pct"]].head(10)

**Observation:**

*(Write what you notice after running the cell above.)*

## 5. Prepare features and target

In [ ]:
target_col = "HeartDisease"

X = df_fe.drop(columns=[target_col])
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 6. Split into train and test sets
`stratify=y` keeps the same class proportions in both sets, which makes evaluation more reliable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)
print()
print("Class proportions (train):", y_train.value_counts(normalize=True).round(3).to_dict())
print("Class proportions (test): ", y_test.value_counts(normalize=True).round(3).to_dict())

**Observation:**

*(Write what you notice after running the cell above.)*

## 7. Preprocessing pipeline
We apply different steps to different column types:

- **Numeric columns:** fill missing values with the **median**, then **standardize** (z-score).
- **Categorical columns:** **One-Hot Encoding** (each category becomes its own 0/1 column).

`handle_unknown="ignore"` prevents an error if the test set contains a category not seen in training.

In [ ]:
numeric_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak", "FastingBS", "maxhr_pct"]
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope", "age_group"]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_pipe, numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

preprocess

## 8. Train and compare models
We compare three classifiers with the **same** preprocessing:

- **Logistic Regression:** simple, interpretable baseline (needs scaled features).
- **K-Nearest Neighbors (KNN):** distance-based (needs scaled features).
- **Random Forest:** tree ensemble, handles mixed features well and gives feature importance.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
}

fitted = {}
rows = []
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

results = pd.DataFrame(rows).set_index("model").round(3)
results

**Observation:**

*(Write what you notice after running the cell above.)*

### 8.1 Cross-validation
A single train/test split can be lucky or unlucky. With **5-fold stratified cross-validation** the training data is split 5 times and the score is averaged.
Because the preprocessing is inside the pipeline, the imputer and scaler are re-fit on each training fold (no leakage).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, pipe in fitted.items():
    acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    rec = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")
    cv_rows.append({"model": name,
                    "cv_accuracy_mean": acc.mean(), "cv_accuracy_std": acc.std(),
                    "cv_recall_mean": rec.mean(),   "cv_recall_std": rec.std()})

pd.DataFrame(cv_rows).set_index("model").round(3)

**Observation:**

*(Write what you notice after running the cell above.)*

## 9. Detailed evaluation of one model
Change `selected` to look at a different model.

In [ ]:
selected = "Random Forest"   # "Logistic Regression", "KNN" or "Random Forest"

model = fitted[selected]
y_pred = model.predict(X_test)

print("Selected model:", selected)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=["No disease (0)", "Disease (1)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No disease", "Disease"],
            yticklabels=["No disease", "Disease"])
plt.title(f"Confusion Matrix - {selected}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

How to read the confusion matrix (rows = actual, columns = predicted):

- top-left: healthy, predicted healthy (correct)
- top-right: healthy, predicted sick (**false positive**)
- bottom-left: sick, predicted healthy (**false negative**, the dangerous mistake)
- bottom-right: sick, predicted sick (correct)

**Observation:**

*(Write what you notice after running the cell above.)*

## 10. Feature importance (Random Forest)
After one-hot encoding, every category becomes its own feature, so we first recover the feature names.

In [ ]:
rf_pipe = fitted["Random Forest"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

fi = (pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False))
fi.head(15)

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=fi.head(15), x="importance", y="feature")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

A high importance means the feature gave useful signal to the model; it does **not** prove a causal relationship.

**Observation:**

*(Write what you notice after running the cell above.)*

---
### Tasks
1. **Does feature engineering help?** Remove `age_group` and `maxhr_pct` from `X` (and from the column lists), re-run, and compare accuracy and recall with the results above.
2. **Tune KNN:** change `n_neighbors` (for example 3, 7, 11) and see how accuracy and recall change.
3. **Final choice:** which model would you choose for this problem and why? Mention recall.